# Batch operations

Single-item calls in RAGU are thin wrappers over the batch versions, never the
other way round — `engine.query()` is literally
`(await engine.batch_query([query]))[0]`. That is not an accident of the
implementation, it is the reason batching is worth using:

- **LLM calls** issued as a batch run concurrently under one shared rate limit
  instead of serializing on each round trip.
- **Embeddings** are additionally batched at the *API* level: 500 texts become one
  HTTP request with `input=[...]`, not 500 requests.
- **Graph reads and writes** collapse into single backend statements. On Neo4j the
  difference between one `UNWIND` and N round trips is the difference between
  seconds and minutes.
- **Query planning** batches across plans: subqueries from unrelated top-level
  questions that are ready at the same step are answered in one call.

Every batch read is index-aligned with its input and maps missing records to `None`
rather than dropping them, so results can always be `zip`-ed back against the
request.

**Environment:** `OPENAI_API_KEY`, `LLM_MODEL_NAME`, `EMBEDDER_MODEL_NAME`, and
optionally `OPENAI_BASE_URL`.

In [ ]:
import os
from pathlib import Path

from ragu import (
    ArtifactsExtractorLLM,
    BuilderArguments,
    KnowledgeGraph,
    LocalSearchEngine,
    Settings,
    SimpleChunker,
)
from ragu.graph.types import Entity, Relation
from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.search_engine.local_search import LocalParams
from ragu.utils.ragu_utils import read_text_from_files

DATA_DIR = Path("data/en")

QUESTIONS = [
    "Who created the C programming language?",
    "Where was Unix developed?",
    "What is Bell Labs known for?",
]

## Models

`batch_size` is how many texts share one HTTP request; `max_concurrent_batches`
caps how many such requests are in flight. Both are set low here so the effect is
visible on a small corpus.

In [ ]:
Settings.language = "english"
Settings.storage_folder = "ragu_working_dir/batch_operations_example"

client = CachedAsyncOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)
llm = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"])
embedder = EmbedderOpenAI(
    client=client,
    model_name=os.environ["EMBEDDER_MODEL_NAME"],
    batch_size=32,
    max_concurrent_batches=4,
)
await embedder.initialize()

## Build the graph

The build itself is batched end to end: chunks are extracted, summarized and
vectorized in batches rather than one at a time. This is the expensive cell.

In [ ]:
knowledge_graph = KnowledgeGraph(
    llm=llm,
    embedder=embedder,
    chunker=SimpleChunker(max_chunk_size=1000),
    artifact_extractor=ArtifactsExtractorLLM(llm=llm, embedder=embedder),
    builder_settings=BuilderArguments(),
)
await knowledge_graph.build_from_docs(read_text_from_files(DATA_DIR))

## Batched LLM calls

`continue_on_error` returns `None` for failed items instead of raising, so one bad
input does not lose the batch.

In [ ]:
answers = await llm.batch_chat_completion(
    [[{"role": "user", "content": f"Answer in one sentence: {q}"}] for q in QUESTIONS],
    desc="LLM batch",
    continue_on_error=True,
)
for question, answer in zip(QUESTIONS, answers):
    print(f"{question} -> {answer}")

## Batched embeddings

In [ ]:
texts = [f"Sentence {index} about programming languages." for index in range(50)]
vectors = await embedder.batch_embed_text(texts, desc="Embedding")

print(f"{len(texts)} texts -> {len(vectors)} vectors "
      f"in ~{-(-len(texts) // embedder.batch_size)} API request(s)")

## Batched retrieval and generation

`batch_search` retrieves without generating — useful for building an evaluation
set, or when the context is fed somewhere other than an LLM. `batch_query` shares
retrieval across the batch and routes generation through one batched LLM call.

In [ ]:
engine = LocalSearchEngine(llm=llm, knowledge_graph=knowledge_graph, embedder=embedder)
params = LocalParams(top_k=8)

for question, retrieval in zip(QUESTIONS, await engine.batch_search(QUESTIONS, params)):
    print(f"{question} -> {len(retrieval.result.entities)} entities")

In [ ]:
for response in await engine.batch_query(QUESTIONS, params):
    print(f"Q: {response.query}")
    print(f"A: {response.response}\n")

## Batched graph writes

One call embeds both descriptions, writes both graph nodes and upserts both
vectors. Per-entity calls would repeat all three round trips.

In [ ]:
thompson = Entity(
    entity_name="Ken Thompson",
    entity_type="PERSON",
    description="Co-creator of Unix and the B programming language.",
    source_chunk_id=["manual-chunk-1"],
)
plan9 = Entity(
    entity_name="Plan 9",
    entity_type="PRODUCT",
    description="Distributed operating system developed at Bell Labs.",
    source_chunk_id=["manual-chunk-1"],
)
relation = Relation(
    subject_id=thompson.id,
    object_id=plan9.id,
    subject_name=thompson.entity_name,
    object_name=plan9.entity_name,
    relation_type="CREATED_BY",
    description="Ken Thompson worked on Plan 9.",
    source_chunk_id=["manual-chunk-1"],
)

await knowledge_graph.upsert_entities([thompson, plan9])
await knowledge_graph.upsert_relations([relation])

## Batched graph reads

Relations are addressed by `EdgeSpec` tuples `(subject, object, relation_id)`.
`relation_id=None` matches every edge between the pair, and each spec gets its own
result list because the graph is a multigraph.

In [ ]:
fetched = await knowledge_graph.get_entities([thompson.id, "ent-missing", plan9.id])
print(f"get_entities:  {[e.entity_name if e else None for e in fetched]}")

edges = await knowledge_graph.get_relations([(thompson.id, plan9.id, None)])
print(f"get_relations: {[len(group) for group in edges]} edge(s) per spec")
print(f"edges_degrees: {await knowledge_graph.edges_degrees([(thompson.id, plan9.id, relation.id)])}")

## Update and delete

Updating replaces the stored record and re-embeds the changed description.

In [ ]:
thompson.description += " Also a co-author of the Go programming language."
await knowledge_graph.update_entities([thompson])

updated = (await knowledge_graph.get_entities([thompson.id]))[0]
print(f"after update: {updated.description}")

In [ ]:
await knowledge_graph.delete_relations([(thompson.id, plan9.id, relation.id)])
await knowledge_graph.delete_entities([thompson.id, plan9.id])

print(f"after delete: {await knowledge_graph.get_entities([thompson.id, plan9.id])}")

In [ ]:
await knowledge_graph.index.close()